# Aula05 — LangChain

Uma seção por vídeo, cada uma autossuficiente. Requer o arquivo `.env` (veja `.example.env`).
Os `main.py` originais continuam em `video0X/`.

## video01

In [ ]:
import operator
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

load_dotenv()

MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL")
if not MODELO_OPENAI_PADRAO:
    raise RuntimeError("Configure OPENAI_CHAT_MODEL no .env")

CATEGORIAS = ("financeiro", "tecnico", "comercial")

def criar_llm():
    return ChatOpenAI(model=os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO))

# ===========================================================================
# 1. O jeito linear (LCEL) — o que as aulas anteriores usaram
# ===========================================================================

def executar_lcel(llm, pergunta: str) -> str:
    """Input -> LLM -> Parse -> Output. Uma via só, sem estado."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Você é um atendente corporativo. Responda em uma frase."),
        ("human", "{pergunta}"),
    ])
    return (prompt | llm | StrOutputParser()).invoke({"pergunta": pergunta})

# ===========================================================================
# 2. O mesmo problema como máquina de estados
# ===========================================================================

class EstadoAtendimento(TypedDict):
    """O contrato de dados do grafo.

    Repare que ele carrega muito mais que 'mensagens': categoria é uma decisão
    intermediária, e trilha é a auditoria de tudo que aconteceu. Num caso real
    entrariam aqui DataFrames, flags de compliance, id do cliente etc.
    """

    pergunta: str
    categoria: str
    resposta: str
    # Annotated + operator.add = REDUCER: o LangGraph SOMA as listas devolvidas
    # por cada nó em vez de sobrescrever. É como o histórico se acumula.
    trilha: Annotated[list[str], operator.add]

def criar_app(llm):
    def classificar(estado: EstadoAtendimento) -> dict:
        """Nó 1: decide a categoria. Devolve SÓ o que mudou."""
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Classifique a pergunta em exatamente uma palavra, entre: "
             f"{', '.join(CATEGORIAS)}. Responda só a palavra."),
            ("human", "{pergunta}"),
        ])
        bruto = (prompt | llm | StrOutputParser()).invoke(
            {"pergunta": estado["pergunta"]}
        ).strip().lower()

        # O LLM é probabilístico: valide a saída antes de confiar nela.
        categoria = bruto if bruto in CATEGORIAS else "comercial"

        return {"categoria": categoria, "trilha": [f"classificar -> {categoria}"]}

    def responder(estado: EstadoAtendimento) -> dict:
        """Nó 2: usa a decisão do nó anterior, que veio pelo estado."""
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Você é um especialista da área {categoria}. Responda em uma frase."),
            ("human", "{pergunta}"),
        ])
        texto = (prompt | llm | StrOutputParser()).invoke({
            "categoria": estado["categoria"],
            "pergunta": estado["pergunta"],
        })
        return {"resposta": texto, "trilha": ["responder"]}

    def auditar(estado: EstadoAtendimento) -> dict:
        """Nó 3: nem todo nó precisa de LLM. Aqui é Python puro."""
        return {"trilha": [f"auditar -> resposta com {len(estado['resposta'])} chars"]}

    grafo = StateGraph(EstadoAtendimento)
    grafo.add_node("classificar", classificar)
    grafo.add_node("responder", responder)
    grafo.add_node("auditar", auditar)

    grafo.add_edge(START, "classificar")
    grafo.add_edge("classificar", "responder")
    grafo.add_edge("responder", "auditar")
    grafo.add_edge("auditar", END)

    return grafo.compile()

def run_aula_5_video_1():
    print("--- AULA 5 / VÍDEO 01: DA CHAIN LINEAR À MÁQUINA DE ESTADOS ---")

    llm = criar_llm()
    pergunta = "Nosso faturamento caiu 8% no trimestre. O que devo olhar primeiro?"

    print("\n=== 1. LCEL: entra string, sai string ===")
    print(f"Pergunta: {pergunta}")
    print(f"Resposta: {executar_lcel(llm, pergunta)}")
    print("\nNão sobrou nada: nenhuma decisão intermediária, nenhum rastro.")

    app = criar_app(llm)

    print("\n=== 2. O desenho do grafo ===")
    print(app.get_graph().draw_mermaid())
    print("Cole isso em https://mermaid.live para ver o diagrama.")

    print("\n=== 3. StateGraph: entra estado, sai estado ===")
    estado_final = app.invoke({
        "pergunta": pergunta,
        "categoria": "",
        "resposta": "",
        "trilha": [],
    })

    print(f"Categoria decidida: {estado_final['categoria']}")
    print(f"Resposta:           {estado_final['resposta']}")
    print("\nTrilha acumulada pelo reducer:")
    for passo in estado_final["trilha"]:
        print(f"  - {passo}")

    print("\n=== 4. Vendo o estado a cada passo ===")
    for parcial in app.stream(
        {"pergunta": pergunta, "categoria": "", "resposta": "", "trilha": []},
        stream_mode="updates",
    ):
        for no, mudanca in parcial.items():
            print(f"  [{no}] devolveu: {list(mudanca.keys())}")

### Execução

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("ERRO: configure OPENAI_API_KEY no .env antes de rodar o script.")
else:
    run_aula_5_video_1()

## video02

In [ ]:
import operator
import os
from pathlib import Path
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

load_dotenv()

MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL")
if not MODELO_OPENAI_PADRAO:
    raise RuntimeError("Configure OPENAI_CHAT_MODEL no .env")

CAMINHO_BANCO = Path(__file__).parent / "memoria_grafo_aula5.db"

MAX_TENTATIVAS = 3

def criar_llm():
    return ChatOpenAI(model=os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO))

class EstadoRedacao(TypedDict):
    tema: str
    rascunho: str
    critica: str
    aprovado: bool
    tentativas: int
    trilha: Annotated[list[str], operator.add]

def criar_app(llm, checkpointer):
    def gerar(estado: EstadoRedacao) -> dict:
        """Escreve (ou reescreve, se já houve crítica)."""
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Escreva UMA frase de resumo executivo sobre o tema. "
             "Se houver crítica anterior, corrija exatamente o que ela apontou."),
            ("human", "Tema: {tema}\nCrítica anterior: {critica}"),
        ])
        texto = (prompt | llm | StrOutputParser()).invoke({
            "tema": estado["tema"],
            "critica": estado["critica"] or "(nenhuma, é a primeira versão)",
        })
        tentativa = estado["tentativas"] + 1
        return {
            "rascunho": texto,
            "tentativas": tentativa,
            "trilha": [f"gerar (tentativa {tentativa})"],
        }

    def revisar(estado: EstadoRedacao) -> dict:
        """O agente critica o próprio trabalho. É a autocrítica do loop."""
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Você revisa resumos executivos. Se o texto tiver UMA frase e citar "
             "um número, responda exatamente APROVADO. Caso contrário, responda "
             "com uma frase dizendo o que corrigir."),
            ("human", "{rascunho}"),
        ])
        parecer = (prompt | llm | StrOutputParser()).invoke(
            {"rascunho": estado["rascunho"]}
        ).strip()

        aprovado = parecer.upper().startswith("APROVADO")
        return {
            "critica": "" if aprovado else parecer,
            "aprovado": aprovado,
            "trilha": [f"revisar -> {'aprovado' if aprovado else 'reprovado'}"],
        }

    def publicar(estado: EstadoRedacao) -> dict:
        """Nó de alta criticidade: é aqui que o grafo vai pausar."""
        return {"trilha": [f"publicar -> {estado['rascunho'][:40]}..."]}

    def decidir(estado: EstadoRedacao) -> str:
        """ARESTA CONDICIONAL: calcula o próximo nó lendo o estado.

        Note o teto de tentativas. Sem ele, um revisor exigente demais deixa
        o grafo girando para sempre — e queimando tokens.
        """
        if estado["aprovado"]:
            return "publicar"
        if estado["tentativas"] >= MAX_TENTATIVAS:
            return "publicar"   # desiste de refinar e segue com o que tem
        return "gerar"          # <- a aresta que volta para trás: o ciclo

    grafo = StateGraph(EstadoRedacao)
    grafo.add_node("gerar", gerar)
    grafo.add_node("revisar", revisar)
    grafo.add_node("publicar", publicar)

    grafo.add_edge(START, "gerar")
    grafo.add_edge("gerar", "revisar")
    grafo.add_conditional_edges("revisar", decidir, ["gerar", "publicar"])
    grafo.add_edge("publicar", END)

    # interrupt_before: o grafo para ANTES de executar 'publicar' e devolve o
    # controle. Nada é publicado sem alguém olhar.
    return grafo.compile(checkpointer=checkpointer, interrupt_before=["publicar"])


def run_aula_5_video_2():
    print("--- AULA 5 / VÍDEO 02: CICLO, PERSISTÊNCIA E HUMAN-IN-THE-LOOP ---")

    llm = criar_llm()
    configuracao = {"configurable": {"thread_id": "redacao_q1"}}
    entrada = {
        "tema": "queda de 8% no faturamento do trimestre",
        "rascunho": "", "critica": "", "aprovado": False,
        "tentativas": 0, "trilha": [],
    }

    with SqliteSaver.from_conn_string(str(CAMINHO_BANCO)) as checkpointer:
        app = criar_app(llm, checkpointer)

        print("\n=== 1. O desenho do grafo (repare na aresta que volta) ===")
        print(app.get_graph().draw_mermaid())

        print("\n=== 2. Rodando até a pausa ===")
        for parcial in app.stream(entrada, config=configuracao, stream_mode="updates"):
            for no, mudanca in parcial.items():
                if no == "__interrupt__":       # o grafo bateu no interrupt_before
                    print("  [PAUSA] interrompido antes de 'publicar'")
                    continue
                print(f"  [{no}] {mudanca['trilha'][0]}")

        print("\n=== 3. O grafo PAUSOU. Estado na mão do humano ===")
        estado = app.get_state(configuracao)
        print(f"  Próximo nó a executar: {estado.next}")
        print(f"  Tentativas até aqui:   {estado.values['tentativas']}")
        print(f"  Rascunho proposto:     {estado.values['rascunho']}")

        print("\n=== 4. OVERRIDE: o humano edita o estado antes de liberar ===")
        app.update_state(
            configuracao,
            {"rascunho": estado.values["rascunho"] + " (revisado por Adriano)"},
        )
        print(f"  Rascunho após edição:  {app.get_state(configuracao).values['rascunho']}")

        print("\n=== 5. APPROVAL: invoke(None) retoma de onde parou ===")
        final = app.invoke(None, config=configuracao)
        print("  Trilha completa:")
        for passo in final["trilha"]:
            print(f"    - {passo}")

        print("\n=== 6. TIME TRAVEL: os snapshots salvos pelo checkpointer ===")
        historico = list(app.get_state_history(configuracao))
        print(f"  {len(historico)} snapshots gravados (do mais recente ao início):")
        for snap in historico[:6]:
            proximo = snap.next or ("FIM",)
            print(f"    próximo={proximo[0]:<12} tentativas={snap.values.get('tentativas')}")
        print("\n  Cada um desses tem um checkpoint_id: dá para reexecutar a partir")
        print("  de qualquer ponto passando-o no config.")

### Execução

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("ERRO: configure OPENAI_API_KEY no .env antes de rodar o script.")
else:
    run_aula_5_video_2()

## video03

In [ ]:
import operator
import os
import time
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

load_dotenv()

MODELO_OPENAI_PADRAO = os.environ.get("OPENAI_CHAT_MODEL")
if not MODELO_OPENAI_PADRAO:
    raise RuntimeError("Configure OPENAI_CHAT_MODEL no .env")

ESPECIALISTAS = ("pesquisador", "analista", "redator")
MAX_PASSOS = 6  # trava contra supervisor indeciso girando para sempre

def criar_llm():
    return ChatOpenAI(model=os.environ.get("OPENAI_CHAT_MODEL", MODELO_OPENAI_PADRAO))

# ===========================================================================
# 1. Padrão Supervisor
# ===========================================================================

class EstadoEquipe(TypedDict):
    tarefa: str
    proximo: str
    passos: int
    entregas: Annotated[list[str], operator.add]
    trilha: Annotated[list[str], operator.add]

def criar_app_supervisor(llm):
    def supervisor(estado: EstadoEquipe) -> dict:
        """Decide quem trabalha agora. Não executa nada — só roteia."""
        ja_feito = [e.split(":")[0] for e in estado["entregas"]]

        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Você coordena uma equipe. Especialistas disponíveis: {equipe}.\n"
             "Já entregaram: {feito}.\n"
             "Responda com UMA palavra: o próximo especialista, ou FINISH se a "
             "tarefa estiver completa. Só a palavra."),
            ("human", "Tarefa: {tarefa}"),
        ])
        bruto = (prompt | llm | StrOutputParser()).invoke({
            "equipe": ", ".join(ESPECIALISTAS),
            "feito": ", ".join(ja_feito) or "ninguém ainda",
            "tarefa": estado["tarefa"],
        }).strip().lower()

        # O supervisor é um LLM: valide a decisão dele contra a lista real.
        proximo = bruto if bruto in ESPECIALISTAS else "FINISH"

        return {
            "proximo": proximo,
            "passos": estado["passos"] + 1,
            "trilha": [f"supervisor -> {proximo}"],
        }

    def criar_especialista(nome: str, instrucao: str):
        def no(estado: EstadoEquipe) -> dict:
            prompt = ChatPromptTemplate.from_messages([
                ("system", instrucao + " Responda em uma frase."),
                ("human", "Tarefa: {tarefa}"),
            ])
            texto = (prompt | llm | StrOutputParser()).invoke(
                {"tarefa": estado["tarefa"]}
            )
            return {"entregas": [f"{nome}: {texto}"], "trilha": [nome]}
        return no

    def rotear(estado: EstadoEquipe) -> str:
        """ARESTA CONDICIONAL: lê a decisão do supervisor e devolve o nó."""
        if estado["passos"] >= MAX_PASSOS:
            return END
        if estado["proximo"] in ESPECIALISTAS:
            return estado["proximo"]
        return END

    grafo = StateGraph(EstadoEquipe)
    grafo.add_node("supervisor", supervisor)
    grafo.add_node("pesquisador",
                   criar_especialista("pesquisador", "Você levanta dados e fontes."))
    grafo.add_node("analista",
                   criar_especialista("analista", "Você interpreta números e aponta causas."))
    grafo.add_node("redator",
                   criar_especialista("redator", "Você escreve o texto final para a diretoria."))

    grafo.add_edge(START, "supervisor")
    grafo.add_conditional_edges("supervisor", rotear, [*ESPECIALISTAS, END])

    for nome in ESPECIALISTAS:
        grafo.add_edge(nome, "supervisor")

    return grafo.compile()

# ===========================================================================
# 2. Latência: sequencial vs paralelo
# ===========================================================================

class EstadoConsulta(TypedDict):
    tarefa: str
    fontes: Annotated[list[str], operator.add]


def _consulta_lenta(nome: str, segundos: float = 1.0):
    """Simula uma chamada de API ou LLM que demora."""
    def no(estado: EstadoConsulta) -> dict:
        time.sleep(segundos)
        return {"fontes": [nome]}
    return no

def criar_app_sequencial():
    grafo = StateGraph(EstadoConsulta)
    grafo.add_node("crm", _consulta_lenta("crm"))
    grafo.add_node("erp", _consulta_lenta("erp"))
    grafo.add_edge(START, "crm")
    grafo.add_edge("crm", "erp")      # erp só começa quando crm termina
    grafo.add_edge("erp", END)
    return grafo.compile()

def criar_app_paralelo():
    grafo = StateGraph(EstadoConsulta)
    grafo.add_node("crm", _consulta_lenta("crm"))
    grafo.add_node("erp", _consulta_lenta("erp"))
    grafo.add_edge(START, "crm")
    grafo.add_edge(START, "erp")      # duas arestas saindo do START = fan-out
    grafo.add_edge("crm", END)
    grafo.add_edge("erp", END)
    return grafo.compile()

def run_aula_5_video_3():
    print("--- AULA 5 / VÍDEO 03: MULTI-AGENTES E ENGENHARIA ---")

    llm = criar_llm()
    app = criar_app_supervisor(llm)

    print("\n=== 1. O desenho da equipe ===")
    print(app.get_graph().draw_mermaid())

    print("\n=== 2. O supervisor distribuindo o trabalho ===")
    entrada = {
        "tarefa": "Explicar a queda de 8% no faturamento do trimestre",
        "proximo": "", "passos": 0, "entregas": [], "trilha": [],
    }
    final = app.invoke(entrada)

    for passo in final["trilha"]:
        print(f"  {passo}")

    print("\n=== 3. O que cada especialista entregou ===")
    for entrega in final["entregas"]:
        print(f"  - {entrega}")

    print("\n=== 4. Latência: dois nós independentes ===")
    entrada_consulta = {"tarefa": "consolidar dados", "fontes": []}

    inicio = time.perf_counter()
    r1 = criar_app_sequencial().invoke(entrada_consulta)
    seq = time.perf_counter() - inicio

    inicio = time.perf_counter()
    r2 = criar_app_paralelo().invoke(entrada_consulta)
    par = time.perf_counter() - inicio

    print(f"  sequencial (crm -> erp): {seq:.2f}s  fontes={r1['fontes']}")
    print(f"  paralelo   (crm | erp):  {par:.2f}s  fontes={r2['fontes']}")
    print("\n  A diferença é só o desenho das arestas. Quando dois nós não")
    print("  dependem um do outro, ligue os dois ao mesmo nó anterior.")

### Execução

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("ERRO: configure OPENAI_API_KEY no .env antes de rodar o script.")
else:
    run_aula_5_video_3()